In [ ]:
# =========================================================
# Notebook 04 — Model Training (Delta Regression Models)
# =========================================================
#
# Purpose:
# Train the core regression models used in the dissertation by
# predicting year-on-year change in Progress 8 (ΔP8) rather
# than absolute Progress 8 levels.
#
# Inputs:
# - school_panel_final.parquet
#
# Stored in:
# - data/processed/
#
# Outputs:
# - delta_xgb_model.joblib
# - delta_model_features.joblib
# - delta_ols_model.joblib
#
# Role in the pipeline:
# This notebook is the model-training stage of the pipeline.
# It trains:
# - an XGBoost delta regressor
# - an OLS delta regression model
#
# These models are later evaluated against the naive
# persistence benchmark described in the dissertation.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
#
# Methodological note:
# The modelling target is:
#   TARGET_P8_DELTA = TARGET_P8 - TARGET_P8_LAG1
#
# This reflects the dissertation’s focus on institutional
# momentum rather than static level prediction.
#
# Leakage control note:
# Lagged Progress 8 (TARGET_P8_LAG1) is used to construct the
# delta target and later reconstruct full P8 forecasts, but it
# is explicitly excluded from the regression feature set.
#
# Validation note:
# This notebook uses an out-of-time split:
# - train years: 2017–2019
# - test years: 2022–2023
#
# This mirrors the temporal validation strategy described in
# the dissertation and avoids random cross-validation leakage.
# =========================================================

# === ONE-CELL: retrain CLEAN delta model (no identifiers, no P8 lag feature) ===

import os
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, precision_score, recall_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer

# =========================================================
# Configuration
# =========================================================
#
# This section defines the file paths used to load the final
# modelling panel and save trained model artefacts.
#
# If reproducing this pipeline on another machine, update
# these paths before running the notebook.
# =========================================================

# -----------------------------
# PATHS (adjust BASE_DIR if needed)
# -----------------------------
BASE_DIR = r"C:\Users\kiero\Documents\msc-dissertation\data"
PROCESSED = os.path.join(BASE_DIR, "processed")

PANEL_PATH = os.path.join(PROCESSED, "school_panel_final.parquet")
DELTA_MODEL_PATH = os.path.join(PROCESSED, "delta_xgb_model.joblib")
DELTA_FEATURES_PATH = os.path.join(PROCESSED, "delta_model_features.joblib")

# --- ADDED: OLS artefact path (optional but useful for reproducibility) ---
OLS_MODEL_PATH = os.path.join(PROCESSED, "delta_ols_model.joblib")

assert os.path.exists(PANEL_PATH), f"Missing panel file: {PANEL_PATH}"

# =========================================================
# Step 1 — Load Final Panel
# =========================================================
#
# The final engineered school-year panel is loaded from the
# previous feature-engineering stage.
# =========================================================

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_parquet(PANEL_PATH).copy()

# Ensure YEAR_START numeric
df["YEAR_START"] = pd.to_numeric(df["YEAR_START"], errors="coerce")

# =========================================================
# Step 2 — Build Delta Target
# =========================================================
#
# The regression task predicts year-on-year change in Progress 8:
#
#   TARGET_P8_DELTA = TARGET_P8 - TARGET_P8_LAG1
#
# This supports the dissertation’s emphasis on forecasting
# institutional momentum rather than direct level prediction.
# =========================================================

# -----------------------------
# BUILD TARGET: delta P8
# -----------------------------
# We predict change in Progress 8: P8_t - P8_(t-1)
for col in ["TARGET_P8", "TARGET_P8_LAG1"]:
    if col not in df.columns:
        raise KeyError(f"Expected column missing: {col}")

df["TARGET_P8"] = pd.to_numeric(df["TARGET_P8"], errors="coerce")
df["TARGET_P8_LAG1"] = pd.to_numeric(df["TARGET_P8_LAG1"], errors="coerce")
df["TARGET_P8_DELTA"] = df["TARGET_P8"] - df["TARGET_P8_LAG1"]

# Keep rows where delta is defined
df = df.dropna(subset=["YEAR_START", "TARGET_P8", "TARGET_P8_LAG1", "TARGET_P8_DELTA"]).copy()

# =========================================================
# Step 3 — Apply Out-of-Time Train/Test Split
# =========================================================
#
# Train years:
# - 2017
# - 2018
# - 2019
#
# Test years:
# - 2022
# - 2023
#
# This preserves temporal ordering and replicates realistic
# deployment conditions.
# =========================================================

# -----------------------------
# TRAIN/TEST SPLIT (Out-of-time)
# -----------------------------
train_years = [2017, 2018, 2019]
test_years = [2022, 2023]

train_df = df[df["YEAR_START"].isin(train_years)].copy()
test_df  = df[df["YEAR_START"].isin(test_years)].copy()

print("Train rows:", train_df.shape, " | Test rows:", test_df.shape)

# =========================================================
# Step 4 — Build Feature Set and Exclude Leakage Variables
# =========================================================
#
# The feature set is restricted to numeric variables and
# excludes:
# - identifiers
# - target variables
# - time/index variables
# - lagged Progress 8
#
# This prevents memorisation and direct autoregressive leakage.
# =========================================================

# -----------------------------
# FEATURE SELECTION (numeric only) + EXCLUSIONS
# -----------------------------
# Explicit identifiers to drop (prevents memorisation / entity lookup)
explicit_id_cols = ["LA NUMBER", "LA NAME", "LA CODE", "SCHOOL NAME"]

# Also exclude anything target-related or time/index leakage
exclude_keywords = [
    "TARGET",      # excludes TARGET_P8, TARGET_P8_LAG1, TARGET_P8_DELTA, etc.
    "YEAR",        # YEAR_START etc.
    "URN",         # URN / URN_FINAL etc.
    "ESTAB",       # establishment identifiers
    "LAESTAB",     # composite identifiers
    "SCHOOL NAME", # string name column
    "LA NAME",     # string LA name column
]

# Start with columns to exclude by keyword
exclude_cols = set()
upper_cols = {c: str(c).upper() for c in df.columns}

for c, cu in upper_cols.items():
    if any(k in cu for k in exclude_keywords):
        exclude_cols.add(c)

# Add explicit id cols if present
exclude_cols.update([c for c in explicit_id_cols if c in df.columns])

# Build numeric feature set
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in numeric_cols if c not in exclude_cols]

# Safety: ensure TARGET_P8_LAG1 is not included
if "TARGET_P8_LAG1" in feature_cols:
    feature_cols.remove("TARGET_P8_LAG1")

# Optional safety: remove any residual label-like columns
for bad in ["TARGET_P8", "TARGET_P8_DELTA"]:
    if bad in feature_cols:
        feature_cols.remove(bad)

# Final X/y
X_train = train_df.reindex(columns=feature_cols)
y_train = train_df["TARGET_P8_DELTA"].astype(float)

X_test = test_df.reindex(columns=feature_cols)
y_test_p8 = test_df["TARGET_P8"].astype(float)
y_test_p8_lag = test_df["TARGET_P8_LAG1"].astype(float)

print("Features:", len(feature_cols))

# =========================================================
# Step 5 — Sanity Checks
# =========================================================
#
# These checks confirm that:
# - identifier variables are excluded
# - lagged P8 is not present in the feature set
# =========================================================

# -----------------------------
# SANITY CHECKS
# -----------------------------
# Confirm identifiers are excluded
still_in = [c for c in explicit_id_cols if c in feature_cols]
print("Identifier cols still in features:", still_in)

# Confirm no P8 lag in features
print("TARGET_P8_LAG1 in features:", "TARGET_P8_LAG1" in feature_cols)

# =========================================================
# Step 6 — Train Delta Regression Models
# =========================================================
#
# Two models are fitted:
# - XGBoost delta regressor
# - OLS delta regressor
#
# XGBoost is used as the primary non-linear model.
# OLS provides a linear benchmark for comparison.
#
# The OLS branch uses median imputation to handle missing
# predictor values.
# =========================================================

# -----------------------------
# TRAIN MODEL (delta regressor)
# -----------------------------
# These are reasonable defaults; if you have tuned params in your notebook, replace them here.
delta_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror",
)

delta_model.fit(X_train, y_train)

imputer = SimpleImputer(strategy="median")
X_train_ols = imputer.fit_transform(X_train)   # fit on train only
X_test_ols  = imputer.transform(X_test)

ols_model = LinearRegression()
ols_model.fit(X_train_ols, y_train)


# =========================================================
# Step 7 — Save Trained Artefacts
# =========================================================
#
# The trained models and final feature list are serialized for
# reuse in evaluation notebooks and the dashboard prototype.
# =========================================================

# -----------------------------
# SAVE MODEL + FEATURES
# -----------------------------
joblib.dump(delta_model, DELTA_MODEL_PATH)
joblib.dump(feature_cols, DELTA_FEATURES_PATH)
print("Saved:", DELTA_MODEL_PATH)
print("Saved:", DELTA_FEATURES_PATH)

# --- ADDED: Save OLS model (optional but recommended) ---
joblib.dump(ols_model, OLS_MODEL_PATH)
print("Saved:", OLS_MODEL_PATH)

# =========================================================
# Outputs Produced
# =========================================================
#
# After successful execution, this notebook produces:
#
# - delta_xgb_model.joblib
# - delta_model_features.joblib
# - delta_ols_model.joblib
#
# These artefacts are used in downstream evaluation and
# dashboard integration.
#
# Next notebook in pipeline:
# - 05_check_features.ipynb
# - 06_final_evaluation.ipynb
# =========================================================

Train rows: (6894, 263)  | Test rows: (7166, 263)
Features: 232
Identifier cols still in features: []
TARGET_P8_LAG1 in features: False


C:\Users\kiero\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['SCHOOL TYPE DESCRIPTION' 'SCHOOL TYPE' 'SCHOOL PHASE'
 'RELIGIOUS CHARACTER' 'GOVERNMENT OFFICE REGION NAME'
 'PARLIAMENTARY CONSTITUENCY' 'LA DISTRICT' 'WARD' 'STATUTORY LOW AGE'
 'STATUTORY HIGH AGE' 'TOTAL SCHOOL WORKFORCE (HEADCOUNT)'
 'TOTAL NUMBER OF CLASSROOM TEACHERS (HEADCOUNT)'
 'TOTAL NUMBER OF TEACHERS IN THE LEADERSHIP GROUP (HEADCOUNT)'
 'TOTAL NUMBER OF TEACHERS (HEADCOUNT)'
 'TOTAL NUMBER OF TEACHING ASSISTANTS (HEADCOUNT)'
 'TOTAL NUMBER OF NON CLASSROOM-BASED SCHOOL SUPPORT STAFF, EXCLUDING AUXILIARY STAFF (HEADCOUNT)'
 'TOTAL NUMBER OF AUXILIARY STAFF (HEADCOUNT)'
 'PERCENTAGE OF ALL TEACHING STAFF WHO WORK PART-TIME (%)'
 'TOTAL SCHOOL WORKFORCE (FULL-TIME EQUIVALENT)'
 'TOTAL NUMBER OF CLASSROOM TEACHERS (FULL-TIME EQUIVALENT)'
 'TOTAL NUMBER OF TEACHERS IN THE LEADERSHIP GROUP (FULL-TIME EQUIVALENT)'
 'TOTAL NUMBE

Saved: C:\Users\kiero\Documents\msc-dissertation\data\processed\delta_xgb_model.joblib
Saved: C:\Users\kiero\Documents\msc-dissertation\data\processed\delta_model_features.joblib
Saved: C:\Users\kiero\Documents\msc-dissertation\data\processed\delta_ols_model.joblib
